## 환경 설정 

In [ ]:
# !pip install -U transformers

In [22]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold

from transformers import AutoProcessor, AutoModelForVision2Seq, Kosmos2ForConditionalGeneration

from trl import SFTTrainer, SFTConfig

from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model

# from trl.commands.cli_utils import SftScriptArguments, TrlParser

In [ ]:
# # !! Important HOTFIX !!
# import transformers.models.kosmos2.modeling_kosmos2 as kosmos2_module

# original_forward_embedding = kosmos2_module.Kosmos2TextTransformer.forward_embedding

# def patched_forward_embedding(self, input_ids, inputs_embeds, image_embeds, img_input_mask, past_key_values_length, position_ids):
#     if inputs_embeds is not None:
#         inputs_embeds = inputs_embeds.clone()
#     return original_forward_embedding(
#         self, input_ids, inputs_embeds, image_embeds,
#         img_input_mask, past_key_values_length, position_ids
#     )

# kosmos2_module.Kosmos2TextTransformer.forward_embedding = patched_forward_embedding


In [2]:
# 환경 설정
warnings.filterwarnings("ignore")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cuda


## 시드고정

In [3]:
# 시드 고정
def seed_everything(seed=47):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

## 모델 선언

In [4]:
model = Kosmos2ForConditionalGeneration.from_pretrained("microsoft/kosmos-2-patch14-224", device_map='auto', torch_dtype=torch.bfloat16, _attn_implementation="sdpa",)
processor = AutoProcessor.from_pretrained("microsoft/kosmos-2-patch14-224")

# 모델 정보
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델 정보:")
print(f"총 파라미터 수: {total_params:,}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


모델 정보:
총 파라미터 수: 1,664,485,376


## 데이터 collator

In [ ]:
class DataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts = []
        images = []
        prompt_lens = []

        for example in examples:
            image = example['image']
            question = example["question"]
            answer = example['answer']
            choices = '\n'.join([f"{chr(65+i)}. {c}" for i, c in enumerate(example['choices'])])

            # prompt 텍스트 생성
            prompt = f"Question: {question}\nChoices: {choices}\nAnswer:"
            full_text = f"{prompt} {answer}"
            texts.append(full_text)
            images.append(image)

            # prompt 길이를 나중에 마스킹에 쓰기 위해 저장
            prompt_input_ids = self.processor.tokenizer(prompt, return_tensors="pt").input_ids[0]
            prompt_lens.append(len(prompt_input_ids))

        # 전체 text + image 처리
        batch = self.processor(images=images, text=texts, padding=True, truncation=True, return_tensors="pt")
        input_ids = batch["input_ids"]

        # labels는 input_ids 복사
        labels = input_ids.clone()

        # 각 sample마다 prompt 길이만큼 -100으로 마스킹
        for i, prompt_len in enumerate(prompt_lens):
            labels[i, :prompt_len] = -100

        # pad 토큰도 무시하도록 마스킹
        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        batch["labels"] = labels
        return batch



data_collator = DataCollator(processor)

## 오피셜 데이터셋

In [6]:
dacon_ds = load_dataset('csv', data_files="../../eg/train.csv", split="train")
example = dacon_ds[1]
example

{'ID': 'TRAIN_001',
 'img_path': './train_input_images/TRAIN_001.jpg',
 'Question': 'What color is the shoe rack in the image?',
 'A': 'Black',
 'B': 'Brown',
 'C': 'White',
 'D': 'Blue',
 'answer': 'B'}

In [7]:
# DACON 오피셜 데이터셋
class DACONDataset(torch.utils.data.Dataset):
    """DACON official dataset."""

    def __init__(self, dataset, processor):
        self.dataset = dataset.remove_columns(["ID"])
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        
        image = Image.open(f"../../eg/{sample['img_path']}").convert('RGB')
        question = sample['Question']
        choices = [sample[c] for c in ['A', 'B', 'C', 'D']]
        answer = sample['answer']
        
        
        return {"image": image, "question": question, "choices": choices, "answer": answer}


dacon_ds = DACONDataset(dacon_ds, processor)
dacon_ds[1]

{'image': <PIL.Image.Image image mode=RGB size=812x812>,
 'question': 'What color is the shoe rack in the image?',
 'choices': ['Black', 'Brown', 'White', 'Blue'],
 'answer': 'B'}

## A-OKVQA 데이터셋

In [53]:
train_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="train")
val_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="validation")
train_ds[1]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=480x640>,
 'question_id': '22ZAvqke8EhGDj8e4eyios',
 'question': 'Where does this man eat pizza?',
 'choices': ['office', 'cafe', 'motel', 'outside'],
 'correct_choice_idx': 0,
 'direct_answers': "['work', 'office', 'work', 'work', 'at work', 'desk', 'at desk', 'office', 'work desk', 'office']",
 'difficult_direct_answer': False,
 'rationales': ['The man is eating pizza at a work desk in an office setting.',
  'The man is near a computer keyboard.',
  'The man is sitting at an office desk at his job and eating his pizza.']}

In [54]:
# A-OKVQA 데이터셋
class AokvqaDataset(torch.utils.data.Dataset):
    """A-OKVQA dataset."""

    def __init__(self, dataset, processor):
        self.dataset = dataset.remove_columns(["question_id", 'direct_answers', 'difficult_direct_answer', 'rationales'])
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        
        question = sample['question']
        choices = sample['choices']
        answer = list(zip(choices, ['A', 'B', 'C', 'D']))[sample['correct_choice_idx']][1]
        image = sample['image'].convert('RGB')
        
        return {"image": image, "question": question, "choices": choices, "answer": answer}

train_ds = AokvqaDataset(train_ds, processor)
val_ds = AokvqaDataset(val_ds, processor)

In [55]:
# train_ds[1]

## visual 7w 데이터셋

In [39]:
# Visual 7w 데이터셋 로드
visual_ds = load_dataset("json", data_files="/mnt/workspace/datasets/visual7w/dataset_v7w_telling.json", split='train')
visual_ds[1]

{'images': {'filename': 'v7w_2359302.jpg',
  'image_id': 2359302,
  'qa_pairs': [{'answer': "Fox's Ginger Biscuits.",
    'image_id': 2359302,
    'multiple_choices': ["Donald's Donuts.",
     "Mac's Macaroni Hut.",
     "Smith's Smothered Chicken Tenders."],
    'qa_id': 252208,
    'question': 'What is written on the white square on the bus?',
    'type': 'what'},
   {'answer': 'Double decker bus.',
    'image_id': 2359302,
    'multiple_choices': ['City bus.', 'School bus.', 'Party bus.'],
    'qa_id': 252209,
    'question': 'What kind of bus is this?',
    'type': 'what'},
   {'answer': 'Day time.',
    'image_id': 2359302,
    'multiple_choices': ['Night time.', 'Evening.', 'Morning.'],
    'qa_id': 252210,
    'question': 'When is this scene taking place?',
    'type': 'when'},
   {'answer': 'Seventy four.',
    'image_id': 2359302,
    'multiple_choices': ['Eight nine.', 'Sixty five.', 'Forty two.'],
    'qa_id': 252214,
    'question': 'What is the number on the very top of th

In [40]:
# 데이터셋 전처리
def preproc_visual7w(dataset):
    processed = []
    for sample in dataset:
        sample = sample['images']
        image_name = sample['filename']

        for i in range(len(sample['qa_pairs'])):
            answer_idx = np.random.randint(0,4)
            choices = sample['qa_pairs'][i]['multiple_choices']
            choices.insert(answer_idx, sample['qa_pairs'][i]['answer'])
            processed.append({'question': sample['qa_pairs'][i]['question'],
                              'answer': ['A', 'B', 'C', 'D'][answer_idx],
                              'answer_idx': answer_idx,
                              'choices': choices,
                              'img_path': f'datasets/visual7w/images/{image_name}'   })
    processed = Dataset.from_list(processed)
    processed = processed.class_encode_column('answer_idx').train_test_split(test_size=0.2, stratify_by_column='answer_idx')
    return processed['train'], processed['test']


train_ds, val_ds = preproc_visual7w(visual_ds)

Casting to class labels: 100%|██████████| 139868/139868 [00:00<00:00, 948214.47 examples/s] 


In [41]:
train_ds[1]

{'question': 'What color is runway?',
 'answer': 'A',
 'answer_idx': 0,
 'choices': ['Gray.', 'Brown.', 'Black.', 'White.'],
 'img_path': 'datasets/visual7w/images/v7w_2393796.jpg'}

In [43]:
# Visual 7w 데이터셋
class VisualDataset(torch.utils.data.Dataset):
    """Visual 7w dataset."""

    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        
        question = sample['question']
        choices = sample['choices']
        answer = sample['answer']
        image = Image.open(''.join(["../../", sample['img_path']])).convert('RGB')
        
        return {"image": image, "question": question, "choices": choices, "answer": answer}

train_ds = VisualDataset(train_ds, processor)
val_ds = VisualDataset(val_ds, processor)

## SFT Trainer

In [ ]:
training_args = SFTConfig(
    report_to='wandb',
    run_name = 'run-0709',
    output_dir="foobar",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy='epochs',
    save_strategy="best",
    optim="adamw_torch_fused",
    bf16=True,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1356338311.py, line 3)

In [57]:
print(model)

Kosmos2ForConditionalGeneration(
  (text_model): Kosmos2TextForCausalLM(
    (model): Kosmos2TextTransformer(
      (embed_tokens): Embedding(65037, 2048, padding_idx=1)
      (embed_positions): Kosmos2TextSinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-23): 24 x Kosmos2TextBlock(
          (self_attn): KosmosTextAttention(
            (k_proj): lora.Linear(
              (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
              (lora_dropout): ModuleDict(
                (default): Dropout(p=0.05, inplace=False)
              )
              (lora_A): ModuleDict(
                (default): Linear(in_features=2048, out_features=16, bias=False)
              )
              (lora_B): ModuleDict(
                (default): Linear(in_features=16, out_features=2048, bias=False)
              )
              (lora_embedding_A): ParameterDict()
              (lora_embedding_B): ParameterDict()
              (lora_magnitude_vector): ModuleDict()

In [ ]:
# # Freeze vision model
# for name, param in model.vision_model.named_parameters():
#     param.requires_grad = False

In [58]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules= 'all-linear',
    exclude_modules=["vision_model"],
    lora_dropout=0.05,
    bias="none",
)

In [61]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dacon_ds,
    peft_config=lora_config,
    data_collator=data_collator,
)

trainer.train()

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


ValueError: Expected input batch_size (3840) to match target batch_size (96).

In [65]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
    data_collator=data_collator,
)

trainer.train()

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss


KeyboardInterrupt: 

## 더미입력으로 확인

In [ ]:
# 더미입력으로 확인
try:
    from PIL import Image
    import numpy as np
    import requests
    
    # PIL 이미지로 더미 이미지 생성 (RGB, 224x224)
    # dummy_image_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
    # dummy_image = Image.fromarray(dummy_image_array, mode='RGB')
    url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
    dummy_image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    dummy_text = "Describe this image."
    
    print("더미 이미지 생성 완료")
    print(f"이미지 크기: {dummy_image.size}, 모드: {dummy_image.mode}")
    
    # 입력 전처리
    inputs = processor(
        images=dummy_image, 
        text=dummy_text, 
        return_tensors="pt"
    )
    
    print("입력 전처리 완료")
    print(f"입력 텐서 키: {list(inputs.keys())}")
    print(f"이미지 텐서 shape: {inputs['pixel_values'].shape}")
    print(f"텍스트 토큰 shape: {inputs['input_ids'].shape}")
    
    # T5 모델을 위한 decoder_input_ids 생성
    # T5는 decoder 시작 시 pad_token_id를 사용
    batch_size = inputs['input_ids'].shape[0]
    decoder_input_ids = torch.full(
        (batch_size, 1), 
        processor.tokenizer.pad_token_id, 
        dtype=torch.long
    )
    inputs['decoder_input_ids'] = decoder_input_ids
    
    print(f"decoder_input_ids 추가: {decoder_input_ids.shape}")
    
    # Forward pass 테스트 (training mode)
    model.train()  # training mode로 설정
    with torch.no_grad():
        outputs = model(**inputs)
    
    print("✓ Forward pass 성공")
    print(f"출력 logits shape: {outputs.logits.shape}")
    print(f"출력 logits 범위: [{outputs.logits.min().item():.4f}, {outputs.logits.max().item():.4f}]")
    
    # Generation 테스트도 수행
    print("\nGeneration 테스트...")
    model.eval()  # evaluation mode로 설정
    with torch.no_grad():
        # decoder_input_ids 제거 (generate에서는 자동 생성)
        gen_inputs = {k: v for k, v in inputs.items() if k != 'decoder_input_ids'}
        generated_ids = model.generate(
            **gen_inputs,
            max_new_tokens=30,
            do_sample=True,
            num_beams=3,
            repetition_penalty=1.5,
        )
    
    # 생성된 텍스트 디코딩
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"✓ Generation 성공")
    print(f"생성된 텍스트 (샘플): '{generated_text}...'")
      
    
except Exception as e:
    print(f"✗ 테스트 실패: {str(e)}")
    # 에러 발생 시 더 자세한 디버깅 정보 제공
    import traceback
    print("상세 에러 정보:")
    traceback.print_exc()


print("\n모델 개조 및 테스트 완료!")

In [ ]:
target_modules = [
    # Q-Former 내의 선형 레이어 (BERT-like 구조)
    "query",            # Q-Former 어텐션의 쿼리 프로젝션
    "key",              # Q-Former 어텐션의 키 프로젝션
    "value",            # Q-Former 어텐션의 값 프로젝션
    "attention.output.dense", # Q-Former 어텐션의 출력 밀집 레이어
    "intermediate.dense", # Q-Former FFN의 중간 밀집 레이어
    "output.dense",       # Q-Former FFN의 출력 밀집 레이어

    # Language Model (Flan-T5-large) 내의 선형 레이어
    "q",                # T5 어텐션의 쿼리 프로젝션
    "k",                # T5 어텐션의 키 프로젝션
    "v",                # T5 어텐션의 값 프로젝션
    "o",                # T5 어텐션의 출력 프로젝션
    "wi_0",             # T5 FFN의 중간 레이어 1
    "wi_1",             # T5 FFN의 중간 레이어 2 (Gated FFN의 경우)
    "wo",               # T5 FFN의 출력 레이어

    # InstructBLIP 특정 레이어
    "language_projection", # 당신이 정의한 Q-Former -> LLM 연결 projection 레이어
    "lm_head"             # 언어 모델의 최종 출력 헤드
]

In [ ]:
# 중복 제거
target_module_names = list(set(target_modules))

lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=target_module_names        
)
model = get_peft_model(model, lora_config)
print(model.print_trainable_parameters())

In [ ]:
# A-OKVQA 데이터셋
train_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="train[:50]")
val_ds = load_dataset("HuggingFaceM4/A-OKVQA", split="validation[:10]")

a = len(train_ds)
# 데이터 유효성 확인
train_ds = train_ds.filter(lambda example: len(example['choices']) == 4)
train_ds = train_ds.filter(lambda example: example['correct_choice_idx'] in [0,1,2,3])
print(f"필터링된 샘플 갯수: {len(train_ds) - a}")

In [ ]:
# A-OKVQA 데이터셋
class AokvqaDataset(torch.utils.data.Dataset):
    """A-OKVQA dataset."""

    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # get image + text
        sample = self.dataset[idx]
        answer_idx = sample['correct_choice_idx']
        answer = sample['choices'][answer_idx]
        image = sample['image'].convert('RGB')
        prompt = f"""
        Based on the image, choose the correct option to the following question.

        Question: {sample['question']}

        Options:
        A. {sample['choices'][0]}
        B. {sample['choices'][1]}
        C. {sample['choices'][2]}
        D. {sample['choices'][3]}

        Answer:
        """
        encoding = self.processor(image, prompt, padding="max_length", max_length=512, truncation=True, return_tensors="pt")
        
        # remove batch dimension
        encoding = {k: v.squeeze() for k, v in encoding.items()}
        label_encoding = self.processor.tokenizer(
            f'{chr(65+answer_idx)}. {answer}',
            max_length=128, # 라벨의 최대 길이 (충분히 크게)
            padding="max_length", # 여기서는 max_length 패딩이 적절
            truncation=True,
            return_tensors="pt"
        )
        encoding["labels"] = label_encoding.input_ids.squeeze(0)
        encoding["labels"][encoding["labels"] == self.processor.tokenizer.pad_token_id] = -100
        return encoding
    

def collate_fn(batch):
    # 'pixel_values', 'qformer_input_ids', 'qformer_attention_mask', 'input_ids', 'attention_mask', 'labels'
    # 이 모든 키들이 배치로 쌓여야 합니다.
    processed_batch = {}
    for key in batch[0].keys():
        processed_batch[key] = torch.stack([sample[key] for sample in batch])
        
    return processed_batch
    
    
train_ds = AokvqaDataset(dataset=train_ds, processor=processor)
val_ds = AokvqaDataset(dataset=val_ds, processor=processor)

## Inference sample

In [ ]:
import requests
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq


model = AutoModelForVision2Seq.from_pretrained("microsoft/kosmos-2-patch14-224")
processor = AutoProcessor.from_pretrained("microsoft/kosmos-2-patch14-224")

# url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/Confusing-Pictures.jpg"
# url = "https://huggingface.co/microsoft/kosmos-2-patch14-224/resolve/main/snowman.png"
url = "https://huggingface.co/microsoft/kosmos-2-patch14-224/resolve/main/two_dogs.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# The original Kosmos-2 demo saves the image first then reload it. For some images, this will give slightly different image input and change the generation outputs.
# image.save("new_image.jpg")
# image = Image.open("new_image.jpg")

inputs = processor(text=prompt, images=image, return_tensors="pt")

generated_ids = model.generate(
    pixel_values=inputs["pixel_values"],
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    image_embeds=None,
    image_embeds_position_mask=inputs["image_embeds_position_mask"],
    use_cache=True,
    max_new_tokens=128,
)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Specify `cleanup_and_extract=False` in order to see the raw model generation.
processed_text = processor.post_process_generation(generated_text, cleanup_and_extract=False)

print(processed_text,'\n')
# `<grounding> An image of<phrase> a snowman</phrase><object><patch_index_0044><patch_index_0863></object> warming himself by<phrase> a fire</phrase><object><patch_index_0005><patch_index_0911></object>.`

# By default, the generated  text is cleanup and the entities are extracted.
processed_text, entities = processor.post_process_generation(generated_text)

print(processed_text,'\n')
# `An image of a snowman warming himself by a fire.`

print(entities,'\n')
# `[('a snowman', (12, 21), [(0.390625, 0.046875, 0.984375, 0.828125)]), ('a fire', (41, 47), [(0.171875, 0.015625, 0.484375, 0.890625)])]`


## Submission

In [ ]:
# 추론
test = pd.read_csv('../../eg/test.csv')
results = []

# 정답 알파벳 추출 함수
def extract_answer_letter(text):
    match = re.search(r"\s*([A-Da-d])\s*", text)
    return match.group(1).upper() if match else "?"


for _, row in tqdm(test.iterrows(), total=len(test)):
    image = Image.open('../../eg/'+row['img_path'])
    choices = '\n'.join([f"{c}. {row[c]}" for c in ['A', 'B', 'C', 'D']])
    
    prompt = f"Question: {row['Question']}. Choose the correct option. \n Options: {choices} \n Answer: "

    # 🔹 전처리
    inputs = processor(images=image, text=prompt, return_tensors="pt").to('cuda')

    # # 🔹 추론
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            # pixel_values=inputs["pixel_values"],
            # input_ids=inputs["input_ids"],
            # attention_mask=inputs["attention_mask"],
            # image_embeds=None,
            # image_embeds_position_mask=inputs["image_embeds_position_mask"],
            # use_cache=True,
            max_new_tokens=128,
        )
    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    generated_text = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=True)[0]

    # By default, the generated  text is cleanup and the entities are extracted.
    processed_text, entities = processor.post_process_generation(generated_text)

    tqdm.write(f'Trimmed output: {processed_text}')

    answer = extract_answer_letter(processed_text)
    results.append(answer)
    tqdm.write(f"Answer: {answer} \n")

print('✅ Done.')

In [ ]:
submission = pd.read_csv('./sample_submission.csv')
submission['answer'] = results
submission.to_csv('./baseline_submit.csv', index=False)
print("Done.")